# Notebook 5 — Hybrid Model Summary

The Hybrid Model combines the outputs of all three previously built models — SVD Collaborative Filtering, Neural Collaborative Filtering and Content Based Filtering — into a single unified recommendation system. The core motivation is that no single model excels at everything. SVD captures collaborative patterns well but is linear. NCF captures non-linear interactions but needs sufficient data. Content Based provides genre-aware recommendations but lacks personalization. By combining all three each model's weaknesses are partially compensated by the others' strengths.

The most critical challenge in building the hybrid was the scale mismatch between models. SVD and NCF produce predicted ratings on a 1–5 scale while Content Based produces accumulated cosine similarity scores ranging from 0 to 382. Combining these directly would cause content based scores to completely dominate the final ranking. This was solved by applying MinMaxScaler to normalize all three model scores to a 0–1 scale independently before combining them with weighted averaging.

This notebook implements the hybrid recommendation system on the MovieLens 1M dataset. The key steps include:

- Loading all saved models — SVD, NCF, user/movie encoders and content based artifacts  
- Applying the same genre preprocessing from Phase 4 to ensure content based scoring works correctly  
- Defining individual scoring functions for each model returning dictionaries of movie ID to score  
- Taking the intersection of movies scored by all three models to ensure every recommendation has signal from all three  
- Normalizing all scores to 0–1 using MinMaxScaler to fix the scale mismatch problem  
- Combining normalized scores using weighted averaging — SVD = 0.4, NCF = 0.4, Content = 0.2  
- Generating top-10 hybrid recommendations for User 1680 with scores from all three models shown  
- Comparing all four models side by side to understand how each thinks differently about the same user  
- Saving the hybrid configuration as `hybrid_config.json` for use in evaluation and deployment phases  

---

## Key Limitation

The intersection approach reduces the movie pool from **3,706 to 1,857 movies**. Movies must be scoreable by all three models to appear in hybrid recommendations — this means roughly half the catalog is invisible to the hybrid model.

---

## Initial Hybrid Weights (Before Tuning)

- **SVD:** 0.40  
- **NCF:** 0.40  
- **Content Based:** 0.20  

These weights were updated after evaluation in **Notebook 6**.

In [1]:
import pandas as pd                          # data manipulation
import numpy as np                           # numerical operations
import pickle                                # loading saved models
import os                                    # file operations
from surprise import SVD                     # load saved SVD model
from tensorflow import keras                 # load saved NCF model
from sklearn.preprocessing import LabelEncoder, MinMaxScaler  # encoding and normalization
from sklearn.metrics.pairwise import cosine_similarity        # content based similarity
import warnings
warnings.filterwarnings('ignore')

print("All libraries loaded successfully!")

All libraries loaded successfully!


# Load Data

In [2]:
base_path = "OneDrive/Documents/Recommendation System"
models_path = f"{base_path}/models"

# Load processed data files from Phase 1
train = pd.read_csv(f'{base_path}/train.csv')
test = pd.read_csv(f'{base_path}/test.csv')
movies = pd.read_csv(f'{base_path}/movies.csv')

print("Train size:", train.shape)
print("Test size:", test.shape)
print("Movies shape:", movies.shape)
print("\nSample train data:")
print(train.head())

Train size: (800167, 11)
Test size: (200042, 11)
Movies shape: (3883, 3)

Sample train data:
   user_id  movie_id  rating            timestamp year_month  \
0     6040       858       4  2000-04-25 23:05:32    2000-04   
1     6040       593       5  2000-04-25 23:05:54    2000-04   
2     6040      2384       4  2000-04-25 23:05:54    2000-04   
3     6040      1961       4  2000-04-25 23:06:17    2000-04   
4     6040      2019       5  2000-04-25 23:06:17    2000-04   

                                               title              genres  \
0                              Godfather, The (1972)  Action|Crime|Drama   
1                   Silence of the Lambs, The (1991)      Drama|Thriller   
2                       Babe: Pig in the City (1998)   Children's|Comedy   
3                                    Rain Man (1988)               Drama   
4  Seven Samurai (The Magnificent Seven) (Shichin...        Action|Drama   

  gender  age  occupation zip_code  
0      M   25           6   

# Load all Saved Models

In [3]:
# --- Load SVD model ---
# SVD was saved as a pickle file in Phase 3
with open(f'{models_path}/svd_model.pkl', 'rb') as f:
    svd_model = pickle.load(f)
print("SVD model loaded!")

# --- Load NCF model ---
# Keras model saved in native .keras format in Phase 5
ncf_model = keras.models.load_model(f'{models_path}/ncf_model.keras')
print("NCF model loaded!")

# --- Load encoders for NCF ---
# Must use exact same encoders used during NCF training
# Different encoders would map IDs to wrong indices
with open(f'{models_path}/user_encoder.pkl', 'rb') as f:
    user_encoder = pickle.load(f)
with open(f'{models_path}/movie_encoder.pkl', 'rb') as f:
    movie_encoder = pickle.load(f)
print("Encoders loaded!")

# --- Load content based models ---
# Cosine similarity matrix computed in Phase 4
cosine_sim = np.load(f'{models_path}/cosine_sim.npy')
print("Cosine similarity matrix loaded!")

# Movie title to index mapping for content based lookups
movie_indices = pd.read_pickle(f'{models_path}/movie_indices.pkl')
print("Movie indices loaded!")

print("\nAll models loaded successfully!")

SVD model loaded!
NCF model loaded!
Encoders loaded!
Cosine similarity matrix loaded!
Movie indices loaded!

All models loaded successfully!


# Genre Preprocessing for Content Based

In [4]:
# Recreate the same genre cleaning from Phase 4
# This ensures content based scoring works correctly
movies['genres_clean'] = movies['genres'].str.replace('|', ' ', regex=False)
movies['genres_clean'] = movies['genres_clean'].str.replace('Sci-Fi', 'SciFi', regex=False)
movies['genres_clean'] = movies['genres_clean'].str.replace('Film-Noir', 'FilmNoir', regex=False)
movies['genres_clean'] = movies['genres_clean'].str.replace("Children's", 'Childrens', regex=False)
movies['genres_clean'] = movies['genres_clean'].fillna('')

print("Genre preprocessing complete!")
print(movies[['title', 'genres', 'genres_clean']].head())

Genre preprocessing complete!
                                title                        genres  \
0                    Toy Story (1995)   Animation|Children's|Comedy   
1                      Jumanji (1995)  Adventure|Children's|Fantasy   
2             Grumpier Old Men (1995)                Comedy|Romance   
3            Waiting to Exhale (1995)                  Comedy|Drama   
4  Father of the Bride Part II (1995)                        Comedy   

                  genres_clean  
0   Animation Childrens Comedy  
1  Adventure Childrens Fantasy  
2               Comedy Romance  
3                 Comedy Drama  
4                       Comedy  


# Individual Score Functions
We separate each model's scoring into its own function. This makes the code clean and testable — we can verify each model's scores independently before combining them. Each function returns a dictionary of {movie_id: score} which makes it easy to align scores across all three models for normalization.

In [5]:
def get_svd_scores(user_id, model, movies_df, rated_movie_ids, n=None):
    """
    Get SVD predicted ratings for all unrated movies for a user.
    Returns a dictionary of {movie_id: predicted_rating}
    """
    all_movie_ids = movies_df['movie_id'].unique()
    
    # Predict rating for every unrated movie
    scores = {}
    for movie_id in all_movie_ids:
        if movie_id not in rated_movie_ids:
            # model.predict returns a Prediction object
            # .est is the estimated rating
            pred = model.predict(user_id, movie_id)
            scores[movie_id] = pred.est
    
    return scores


def get_ncf_scores(user_id, model, movies_df, rated_movie_ids,
                   user_encoder, movie_encoder):
    """
    Get NCF predicted ratings for all unrated movies for a user.
    Returns a dictionary of {movie_id: predicted_rating}
    """
    # Check if user exists in encoder
    if user_id not in user_encoder.classes_:
        return {}
    
    # Encode user ID to integer index
    user_idx = user_encoder.transform([user_id])[0]
    
    # Get unrated movies that exist in encoder
    all_movie_ids = movies_df['movie_id'].unique()
    unrated_movies = [
        m for m in all_movie_ids
        if m not in rated_movie_ids
        and m in movie_encoder.classes_
    ]
    
    # Encode movie IDs
    movie_indices_arr = movie_encoder.transform(unrated_movies)
    
    # Create input arrays — repeat user index for every movie
    user_arr = np.array([user_idx] * len(unrated_movies))
    movie_arr = np.array(movie_indices_arr)
    
    # Predict in one batch — much faster than one at a time
    predictions = model.predict(
        [user_arr, movie_arr],
        batch_size=512,
        verbose=0
    ).flatten()
    
    # Convert from 0-1 back to 1-5 scale
    predictions = predictions * 4 + 1
    predictions = np.clip(predictions, 1, 5)
    
    # Return as dictionary
    return dict(zip(unrated_movies, predictions))


def get_content_scores(user_id, train_df, movies_df,
                       cosine_sim, movie_indices, rated_movie_ids):
    """
    Get accumulated content based scores for all unrated movies for a user.
    Returns a dictionary of {movie_id: accumulated_cosine_score}
    """
    # Get movies this user rated highly (4 or 5 stars)
    user_ratings = train_df[
        (train_df['user_id'] == user_id) &
        (train_df['rating'] >= 4)
    ].copy()
    
    if user_ratings.empty:
        return {}
    
    # Merge to get movie titles
    user_ratings = pd.merge(
        user_ratings[['user_id', 'movie_id', 'rating']],
        movies_df[['movie_id', 'title']],
        on='movie_id',
        how='inner'
    )
    
    # Initialize score array — one slot per movie
    scores = np.zeros(len(movies_df))
    
    # Accumulate similarity scores from each highly rated movie
    for _, row in user_ratings.iterrows():
        title = row['title']
        if title in movie_indices:
            idx = movie_indices[title]
            # Add similarity scores of this movie to all others
            scores += cosine_sim[idx]
    
    # Map scores back to movie IDs
    result = {}
    for i, movie_id in enumerate(movies_df['movie_id'].values):
        if movie_id not in rated_movie_ids:
            result[movie_id] = scores[i]
    
    return result

print("All scoring functions defined successfully!")

All scoring functions defined successfully!


# Normalize and Combine Scores
 score normalization. Without this content based scores would completely dominate the hybrid because they're on a scale of 0–382 while SVD and NCF are on 1–5. MinMaxScaler maps each model's scores to 0–1 independently so a score of 0.9 means "this model strongly recommends this movie" for all three models equally. The weights then control how much influence each model has on the final ranking.

In [6]:
def get_hybrid_recommendations(user_id, svd_model, ncf_model, train_df,
                                movies_df, user_encoder, movie_encoder,
                                cosine_sim, movie_indices,
                                svd_weight=0.4, ncf_weight=0.4,
                                content_weight=0.2, n=10):
    """
    Generate hybrid recommendations by combining SVD, NCF and content based scores.
    
    Weights must sum to 1.0:
    - svd_weight=0.4: SVD gets highest weight (best performing model)
    - ncf_weight=0.4: NCF gets equal weight to SVD
    - content_weight=0.2: Content based gets lower weight (less precise)
    
    Returns top-N recommended movies with combined scores.
    """
    
    # Get all movies this user has already rated
    # Used to filter out already seen movies from all three models
    rated_movie_ids = set(
        train_df[train_df['user_id'] == user_id]['movie_id'].values
    )
    
    print(f"Getting scores for User {user_id}...")
    
    # --- Step 1: Get raw scores from each model ---
    print("  Getting SVD scores...")
    svd_scores = get_svd_scores(
        user_id, svd_model, movies_df, rated_movie_ids
    )
    
    print("  Getting NCF scores...")
    ncf_scores = get_ncf_scores(
        user_id, ncf_model, movies_df, rated_movie_ids,
        user_encoder, movie_encoder
    )
    
    print("  Getting Content Based scores...")
    content_scores = get_content_scores(
        user_id, train_df, movies_df,
        cosine_sim, movie_indices, rated_movie_ids
    )
    
    # --- Step 2: Find common movies across all three models ---
    # Only recommend movies that all three models can score
    # This ensures fair comparison and combination
    common_movies = set(svd_scores.keys()) & \
                    set(ncf_scores.keys()) & \
                    set(content_scores.keys())
    
    print(f"  Common movies across all models: {len(common_movies)}")
    
    if len(common_movies) == 0:
        print("No common movies found across all models")
        return None
    
    # --- Step 3: Convert to aligned arrays ---
    # Sort movie IDs for consistent ordering
    movie_list = sorted(list(common_movies))
    
    # Extract scores in same order for all three models
    svd_arr = np.array([svd_scores[m] for m in movie_list]).reshape(-1, 1)
    ncf_arr = np.array([ncf_scores[m] for m in movie_list]).reshape(-1, 1)
    content_arr = np.array([content_scores[m] for m in movie_list]).reshape(-1, 1)
    
    # --- Step 4: Normalize all scores to 0-1 scale ---
    # This is the critical fix for the scale mismatch problem
    # SVD scores: 1-5 range
    # NCF scores: 1-5 range
    # Content scores: 0-382 range
    # After MinMaxScaler: all scores are 0-1 range
    scaler = MinMaxScaler()
    
    # fit_transform learns min/max and scales in one step
    svd_norm = scaler.fit_transform(svd_arr).flatten()
    ncf_norm = scaler.fit_transform(ncf_arr).flatten()
    content_norm = scaler.fit_transform(content_arr).flatten()
    
    # --- Step 5: Combine normalized scores with weights ---
    # Higher weight = that model has more influence on final recommendation
    # SVD: 0.4 — best performing model, gets highest influence
    # NCF: 0.4 — captures non-linear patterns, equal to SVD
    # Content: 0.2 — genre based, less precise, lower influence
    hybrid_scores = (svd_weight * svd_norm +
                     ncf_weight * ncf_norm +
                     content_weight * content_norm)
    
    # --- Step 6: Build results dataframe ---
    results = pd.DataFrame({
        'movie_id': movie_list,
        'hybrid_score': hybrid_scores,
        'svd_score': svd_arr.flatten(),       # original SVD score (1-5)
        'ncf_score': ncf_arr.flatten(),       # original NCF score (1-5)
        'content_score': content_arr.flatten() # original content score (0-382)
    })
    
    # Merge with movie titles and genres
    results = results.merge(
        movies_df[['movie_id', 'title', 'genres']],
        on='movie_id',
        how='left'
    )
    
    # Sort by hybrid score and return top N
    results = results.sort_values('hybrid_score', ascending=False).head(n)
    
    return results[['title', 'genres', 'hybrid_score',
                     'svd_score', 'ncf_score',
                     'content_score']].reset_index(drop=True)


# Test with User 1680 — our consistent test subject across all phases
active_users = train['user_id'].value_counts()
test_user = active_users.index[0]  # User 1680

print(f"Generating Hybrid Recommendations for User {test_user}...\n")
hybrid_recs = get_hybrid_recommendations(
    user_id=test_user,
    svd_model=svd_model,
    ncf_model=ncf_model,
    train_df=train,
    movies_df=movies,
    user_encoder=user_encoder,
    movie_encoder=movie_encoder,
    cosine_sim=cosine_sim,
    movie_indices=movie_indices,
    n=10
)

print("\nTop 10 Hybrid Recommendations for User 1680:")
print(hybrid_recs)

Generating Hybrid Recommendations for User 1680...

Getting scores for User 1680...
  Getting SVD scores...
  Getting NCF scores...
  Getting Content Based scores...
  Common movies across all models: 1857

Top 10 Hybrid Recommendations for User 1680:
                                            title                genres  \
0                         Apple, The (Sib) (1998)                 Drama   
1                           Cool Hand Luke (1967)          Comedy|Drama   
2                            Almost Famous (2000)          Comedy|Drama   
3                              City Lights (1931)  Comedy|Drama|Romance   
4                                 Lamerica (1994)                 Drama   
5      Life Is Beautiful (La Vita è bella) (1997)          Comedy|Drama   
6              Cold Fever (Á köldum klaka) (1994)          Comedy|Drama   
7                Celebration, The (Festen) (1998)                 Drama   
8                Shawshank Redemption, The (1994)                 Drama  

# Compare all Models Side by Side

In [7]:
def get_model_comparison(user_id, train_df, movies_df):
    """
    Show recommendations from all four models side by side for comparison.
    """
    rated_movie_ids = set(
        train_df[train_df['user_id'] == user_id]['movie_id'].values
    )
    
    # --- SVD recommendations ---
    svd_scores = get_svd_scores(user_id, svd_model, movies_df, rated_movie_ids)
    svd_top = sorted(svd_scores.items(), key=lambda x: x[1], reverse=True)[:10]
    svd_titles = [movies_df[movies_df['movie_id'] == m]['title'].values[0]
                  for m, _ in svd_top]
    
    # --- NCF recommendations ---
    ncf_scores = get_ncf_scores(user_id, ncf_model, movies_df, rated_movie_ids,
                                 user_encoder, movie_encoder)
    ncf_top = sorted(ncf_scores.items(), key=lambda x: x[1], reverse=True)[:10]
    ncf_titles = [movies_df[movies_df['movie_id'] == m]['title'].values[0]
                  for m, _ in ncf_top]
    
    # --- Content Based recommendations ---
    content_scores = get_content_scores(user_id, train_df, movies_df,
                                         cosine_sim, movie_indices, rated_movie_ids)
    content_top = sorted(content_scores.items(), key=lambda x: x[1], reverse=True)[:10]
    content_titles = [movies_df[movies_df['movie_id'] == m]['title'].values[0]
                      for m, _ in content_top]
    
    # --- Hybrid recommendations ---
    hybrid = get_hybrid_recommendations(
        user_id, svd_model, ncf_model, train_df, movies_df,
        user_encoder, movie_encoder, cosine_sim, movie_indices, n=10
    )
    hybrid_titles = hybrid['title'].tolist()
    
    # --- Print all four side by side ---
    print(f"{'Rank':<6} {'SVD':<35} {'NCF':<35} {'Content':<35} {'Hybrid':<35}")
    print("-" * 111)
    for i in range(10):
        svd_t = svd_titles[i][:33] if i < len(svd_titles) else ""
        ncf_t = ncf_titles[i][:33] if i < len(ncf_titles) else ""
        con_t = content_titles[i][:33] if i < len(content_titles) else ""
        hyb_t = hybrid_titles[i][:33] if i < len(hybrid_titles) else ""
        print(f"{i+1:<6} {svd_t:<35} {ncf_t:<35} {con_t:<35} {hyb_t:<35}")

print(f"\n=== Model Comparison for User {test_user} ===\n")
get_model_comparison(test_user, train, movies)


=== Model Comparison for User 1680 ===

Getting scores for User 1680...
  Getting SVD scores...
  Getting NCF scores...
  Getting Content Based scores...
  Common movies across all models: 1857
Rank   SVD                                 NCF                                 Content                             Hybrid                             
---------------------------------------------------------------------------------------------------------------
1      Apple, The (Sib) (1998)             Shawshank Redemption, The (1994)    Waiting to Exhale (1995)            Apple, The (Sib) (1998)            
2      Lamerica (1994)                     Sanjuro (1962)                      Kicking and Screaming (1995)        Cool Hand Luke (1967)              
3      Paths of Glory (1957)               Paths of Glory (1957)               Big Bully (1996)                    Almost Famous (2000)               
4      Julien Donkey-Boy (1999)            Great Escape, The (1963)            Last Summe

# Save Hybrid Model Components

In [8]:
# Save the hybrid weights configuration
# These weights can be tuned in Phase 7 based on evaluation metrics
hybrid_config = {
    'svd_weight': 0.4,
    'ncf_weight': 0.4,
    'content_weight': 0.2
}

import json
with open(f'{base_path}/hybrid_config.json', 'w') as f:
    json.dump(hybrid_config, f)

print("Hybrid configuration saved!")
print(f"Weights: SVD={hybrid_config['svd_weight']}, "
      f"NCF={hybrid_config['ncf_weight']}, "
      f"Content={hybrid_config['content_weight']}")
print("\nAll hybrid components are the individual model files already saved.")
print("No additional model files needed for hybrid.")

Hybrid configuration saved!
Weights: SVD=0.4, NCF=0.4, Content=0.2

All hybrid components are the individual model files already saved.
No additional model files needed for hybrid.
